<a href="https://colab.research.google.com/github/faww02/API/blob/main/Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
<!DOCTYPE html>
<html lang="id"><head><meta charset="utf-8"><title>Simulator</title></head>
<body>
  <h2>Simulator Dukungan CT</h2>
  <div id="app"></div>
  <script>
    // semua JavaScript ditulis di sini
  </script>
</body></html>

In [ ]:
const app = document.getElementById("app");
function start(){
  app.innerHTML = `<input type="text" id="sid" placeholder="ID siswa">
                   <button onclick="begin()">Mulai</button>`;
}
start();

In [ ]:
const ITEMS = [
 {pillar:"Dekomposisi", type:"mc", q:"Pertanyaan...", options:["A","B","C","D"], answer:1},
 {pillar:"Abstraksi",  type:"short", q:"Hasil luas_persegi(5)?", answer:"25"},
 {pillar:"Abstraksi",  type:"trace", q:"Keluaran kode?", code:"print(3+1)", answer:"4"}
];

In [ ]:
let state = {idx:0, log:[], sid:"", t0:0, revisions:0, current:null};

function begin(){
  state.sid = document.getElementById("sid").value || "anon";
  state.idx = 0; state.log = [];
  showItem();
}
function showItem(){
  const it = ITEMS[state.idx];
  state.t0 = performance.now();      // stopwatch mulai
  state.revisions = 0; state.current = null;
  // ...bangun HTML soal, lalu app.innerHTML = ...
}

In [ ]:
function changed(v){
  if(state.current !== null && String(v) !== String(state.current)) state.revisions++;
  state.current = v;
}

In [ ]:
function submitItem(){
  const it = ITEMS[state.idx];
  const sec = (performance.now() - state.t0) / 1000;      // detik
  const benar = /* bandingkan jawaban dengan it.answer */;
  state.log.push({item:state.idx+1, time:sec, revisions:state.revisions, correct:benar?1:0});
  state.idx++;
  state.idx < ITEMS.length ? showItem() : finish();
}

In [ ]:
function finish(){
  const n = state.log.length;
  const sum = k => state.log.reduce((a,b)=>a+b[k],0);
  const feat = {
    completion_time:  sum("time") / n,              // rata-rata detik/butir
    answer_revisions: sum("revisions") / n,         // rata-rata revisi/butir
    accuracy_score:   sum("correct") / n * 100      // persen benar
  };
  ...
}

In [ ]:
const MODEL = { mean:[...], std:[...], centroids_z:[[...],[...],[...]] };  // dari centroids.json

function dispatchTier(f){
  const x = [f.completion_time, f.answer_revisions, f.accuracy_score]
              .map((v,i) => (v - MODEL.mean[i]) / MODEL.std[i]);   // standarisasi
  let best = 0, bestD = Infinity;
  MODEL.centroids_z.forEach((c,i) => {
    const d = Math.hypot(...c.map((cv,j) => cv - x[j]));           // jarak Euclidean
    if(d < bestD){ bestD = d; best = i; }
  });
  return best + 1;                                                 // tier 1/2/3
}

In [ ]:
const SUPPORT = {1:{nama:"Tier 1...", isi:"..."}, 2:{...}, 3:{...}};
const tier = dispatchTier(feat);
app.innerHTML = `<p>${SUPPORT[tier].nama}</p><pre>${SUPPORT[tier].isi}</pre>`;

In [ ]:
const CONFIG = { APPS_SCRIPT_URL: "const CONFIG = { APPS_SCRIPT_URL: "https://script.google.com/macros/s/AKfycbzxjN8DqaXmPw64zhi41-0B2aiCxww4s-QhuEuLdae4esLnDZlGSLBImOCMS4tIbp5OWA/exec", MODEL: {...} };", MODEL: {...} };

In [ ]:
fetch(CONFIG.APPS_SCRIPT_URL, {
  method: "POST", mode: "no-cors",
  headers: {"Content-Type": "text/plain"},
  body: JSON.stringify(payload)
});